### Inventory Turnover Ratio
- Inventory Turnover = Cost of Goods Sold / Average Inventory

### Safety Stock
- Safety Stock = Z × σ × √Lead Time

Where Z = Service Level Factor

σ = Standard deviation of demand

Lead Time = supplier delivery time

### Reorder Point
- Reorder Point = Average Daily Demand × Lead Time + Safety Stock

### EOQ (Economic Order Quantity)
- EOQ = √((2DS)/H)

Where
D = Annual Demand

S = Ordering Cost

H = Holding Cost

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_csv("Walmart_Cleaned_Data.csv")

df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size,Year,Month,Week,Quarter,Day
0,1,1,2010-02-05,24924.50,False,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,A,151315,2010,2,5,1,5
1,1,1,2010-02-12,46039.49,True,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,A,151315,2010,2,6,1,12
2,1,1,2010-02-19,41595.55,False,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,A,151315,2010,2,7,1,19
3,1,1,2010-02-26,19403.54,False,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,A,151315,2010,2,8,1,26
4,1,1,2010-03-05,21827.90,False,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,A,151315,2010,3,9,1,5


In [4]:
#Create Department-wise Demand Summary
demand = df.groupby(['Store','Dept']).agg(
    Average_Weekly_Demand=('Weekly_Sales','mean'),
    Demand_STD=('Weekly_Sales','std'),
    Maximum_Demand=('Weekly_Sales','max'),
    Minimum_Demand=('Weekly_Sales','min'),
    Total_Sales=('Weekly_Sales','sum')
).reset_index()

demand.head()

,Store,Dept,Average_Weekly_Demand,Demand_STD,Maximum_Demand,Minimum_Demand,Total_Sales
0,1,1,22513.322937,9854.349032,57592.12,14537.37,3219405.18
1,1,2,46102.090420,3440.673222,65615.36,35819.83,6592598.93
2,1,3,13150.478042,8708.978853,51159.17,6165.73,1880518.36
3,1,4,36964.154476,2930.698313,47893.23,32497.43,5285874.09
4,1,5,24257.941119,11330.286495,85676.09,11570.27,3468885.58


In [5]:
#Assume Lead Time
LEAD_TIME = 2        # weeks
SERVICE_LEVEL = 1.65 # 95% service level

In [6]:
#Calculate Safety Stock
demand['Safety_Stock'] = (
    SERVICE_LEVEL
    * demand['Demand_STD']
    * np.sqrt(LEAD_TIME)
)

demand.head()

,Store,Dept,Average_Weekly_Demand,Demand_STD,Maximum_Demand,Minimum_Demand,Total_Sales,Safety_Stock
0,1,1,22513.322937,9854.349032,57592.12,14537.37,3219405.18,22994.654182
1,1,2,46102.090420,3440.673222,65615.36,35819.83,6592598.93,8028.647112
2,1,3,13150.478042,8708.978853,51159.17,6165.73,1880518.36,20321.987413
3,1,4,36964.154476,2930.698313,47893.23,32497.43,5285874.09,6838.644949
4,1,5,24257.941119,11330.286495,85676.09,11570.27,3468885.58,26438.683965


In [7]:
#Calculate Reorder Point
demand['Reorder_Point'] = (
    demand['Average_Weekly_Demand']*LEAD_TIME
    + demand['Safety_Stock']
)

In [8]:
#Simulate Current Inventory
np.random.seed(42)

demand['Current_Inventory'] = np.random.randint(
    500,
    7000,
    len(demand)
)

demand.head()

,Store,Dept,Average_Weekly_Demand,Demand_STD,Maximum_Demand,Minimum_Demand,Total_Sales,Safety_Stock,Reorder_Point,Current_Inventory
0,1,1,22513.322937,9854.349032,57592.12,14537.37,3219405.18,22994.654182,68021.300056,1360
1,1,2,46102.090420,3440.673222,65615.36,35819.83,6592598.93,8028.647112,100232.827951,5890
2,1,3,13150.478042,8708.978853,51159.17,6165.73,1880518.36,20321.987413,46622.943497,5726
3,1,4,36964.154476,2930.698313,47893.23,32497.43,5285874.09,6838.644949,80766.953900,5691
4,1,5,24257.941119,11330.286495,85676.09,11570.27,3468885.58,26438.683965,74954.566203,4272


In [9]:
#Identify Stockout Risk
demand['Stockout_Risk'] = np.where(
    demand['Current_Inventory'] <
    demand['Reorder_Point'],
    "High",
    "Low"
)

demand.head()

,Store,Dept,Average_Weekly_Demand,Demand_STD,Maximum_Demand,Minimum_Demand,Total_Sales,Safety_Stock,Reorder_Point,Current_Inventory,Stockout_Risk
0,1,1,22513.322937,9854.349032,57592.12,14537.37,3219405.18,22994.654182,68021.300056,1360,High
1,1,2,46102.090420,3440.673222,65615.36,35819.83,6592598.93,8028.647112,100232.827951,5890,High
2,1,3,13150.478042,8708.978853,51159.17,6165.73,1880518.36,20321.987413,46622.943497,5726,High
3,1,4,36964.154476,2930.698313,47893.23,32497.43,5285874.09,6838.644949,80766.953900,5691,High
4,1,5,24257.941119,11330.286495,85676.09,11570.27,3468885.58,26438.683965,74954.566203,4272,High


In [10]:
#Overstock Risk
demand['Overstock_Risk'] = np.where(
    demand['Current_Inventory'] >
    demand['Average_Weekly_Demand']*6,
    "High",
    "Low"
)

In [11]:
#Inventory Turnover Ratio
demand['Inventory_Turnover'] = (
    demand['Total_Sales']
    /
    demand['Current_Inventory']
)

In [12]:
#EOQ
demand['Annual_Demand'] = (
    demand['Average_Weekly_Demand']*52
)

demand['EOQ'] = np.sqrt(
    (2*demand['Annual_Demand']*500)/400) # assuming order cost =500, holding cost=40

demand.head()

C:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


,Store,Dept,Average_Weekly_Demand,Demand_STD,Maximum_Demand,Minimum_Demand,Total_Sales,Safety_Stock,Reorder_Point,Current_Inventory,Stockout_Risk,Overstock_Risk,Inventory_Turnover,Annual_Demand,EOQ
0,1,1,22513.322937,9854.349032,57592.12,14537.37,3219405.18,22994.654182,68021.300056,1360,High,Low,2367.209691,1.170693e+06,1710.769412
1,1,2,46102.090420,3440.673222,65615.36,35819.83,6592598.93,8028.647112,100232.827951,5890,High,Low,1119.286745,2.397309e+06,2448.115960
2,1,3,13150.478042,8708.978853,51159.17,6165.73,1880518.36,20321.987413,46622.943497,5726,High,Low,328.417457,6.838249e+05,1307.502254
3,1,4,36964.154476,2930.698313,47893.23,32497.43,5285874.09,6838.644949,80766.953900,5691,High,Low,928.812878,1.922136e+06,2192.108593
4,1,5,24257.941119,11330.286495,85676.09,11570.27,3468885.58,26438.683965,74954.566203,4272,High,Low,812.005051,1.261413e+06,1775.818782


In [13]:
#Inventory Dashboard Table
inventory = demand[[
    'Store',
    'Dept',
    'Average_Weekly_Demand',
    'Safety_Stock',
    'Reorder_Point',
    'Current_Inventory',
    'Inventory_Turnover',
    'EOQ',
    'Stockout_Risk',
    'Overstock_Risk'
]]

inventory.head()

,Store,Dept,Average_Weekly_Demand,Safety_Stock,Reorder_Point,Current_Inventory,Inventory_Turnover,EOQ,Stockout_Risk,Overstock_Risk
0,1,1,22513.322937,22994.654182,68021.300056,1360,2367.209691,1710.769412,High,Low
1,1,2,46102.090420,8028.647112,100232.827951,5890,1119.286745,2448.115960,High,Low
2,1,3,13150.478042,20321.987413,46622.943497,5726,328.417457,1307.502254,High,Low
3,1,4,36964.154476,6838.644949,80766.953900,5691,928.812878,2192.108593,High,Low
4,1,5,24257.941119,26438.683965,74954.566203,4272,812.005051,1775.818782,High,Low


In [14]:
#High Stockout Risk
high_stockout = inventory[
    inventory['Stockout_Risk']=="High"
]

high_stockout.head()

,Store,Dept,Average_Weekly_Demand,Safety_Stock,Reorder_Point,Current_Inventory,Inventory_Turnover,EOQ,Stockout_Risk,Overstock_Risk
0,1,1,22513.322937,22994.654182,68021.300056,1360,2367.209691,1710.769412,High,Low
1,1,2,46102.090420,8028.647112,100232.827951,5890,1119.286745,2448.115960,High,Low
2,1,3,13150.478042,20321.987413,46622.943497,5726,328.417457,1307.502254,High,Low
3,1,4,36964.154476,6838.644949,80766.953900,5691,928.812878,2192.108593,High,Low
4,1,5,24257.941119,26438.683965,74954.566203,4272,812.005051,1775.818782,High,Low


In [15]:
#High Overstock Risk
high_overstock = inventory[
    inventory['Overstock_Risk']=="High"
]

high_overstock.head()

,Store,Dept,Average_Weekly_Demand,Safety_Stock,Reorder_Point,Current_Inventory,Inventory_Turnover,EOQ,Stockout_Risk,Overstock_Risk
26,1,28,593.113986,495.711345,1681.939317,5055,16.778497,277.677544,Low,High
41,1,45,23.995070,41.600934,89.591075,3505,0.486063,55.851223,Low,High
43,1,47,-116.454545,679.956258,447.047167,2399,-1.067945,NaN,Low,High
46,1,51,32.712727,83.268123,148.693578,3702,0.291605,65.212380,Low,High
48,1,54,90.921053,232.316584,414.158690,4390,2.754556,108.718613,Low,High


In [16]:
#Export
inventory.to_csv(
    "Inventory_Optimization.csv",
    index=False
)

print("Inventory file exported successfully.")

Inventory file exported successfully.


In [17]:
#KPI Summary
print("="*40)

print("Total Departments :",len(inventory))

print()

print("High Stockout Risk :",
      (inventory['Stockout_Risk']=="High").sum())

print()

print("High Overstock Risk :",
      (inventory['Overstock_Risk']=="High").sum())

print()

print("Average Inventory Turnover :",
      round(inventory['Inventory_Turnover'].mean(),2))

print("="*40)

Total Departments : 3331

High Stockout Risk : 2499

High Overstock Risk : 703

Average Inventory Turnover : 816.31


- Safety Stock

Safety stock was calculated to account for uncertainty in weekly demand, ensuring product availability during unexpected demand surges.

- Reorder Point

Reorder points were computed using average demand, lead time, and safety stock, enabling timely replenishment before inventory depletion.

- Stockout Risk

Departments where current inventory falls below the reorder point were classified as high stockout risk, indicating an immediate need for replenishment.

- Overstock Risk

Departments maintaining inventory levels significantly above expected demand were classified as overstock risk, highlighting opportunities to reduce holding costs.

- Inventory Turnover

Inventory turnover was estimated to evaluate inventory efficiency. Higher turnover suggests effective inventory utilization, while lower turnover may indicate slow-moving stock.

- EOQ

Economic Order Quantity was calculated to recommend an optimal order size that balances ordering costs and inventory holding costs.


- The Walmart dataset primarily contained historical sales information. To demonstrate inventory optimization, I derived inventory metrics using forecasted demand and standard inventory management formulas. I documented assumptions for lead time, service level, ordering cost, holding cost, and current inventory. These assumptions enabled the calculation of safety stock, reorder points, EOQ, inventory turnover, and stockout risk, which are commonly used to support inventory planning.